In [ ]:
# Install required packages in Google Colab
!pip install -q pandas numpy scikit-learn plotly gradio


In [ ]:
# Essie M.
# CELL 1 — Install and import packages
# Run this cell first in Google Colab.

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import gradio as gr
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print("Gradio version:", gr.__version__)


# CELL 2 — Load, validate, and prepare the Airbnb dataset

FILE_NAME = "Airbnb_db.csv"
FILE_PATH = Path(FILE_NAME)

required_columns = [
    "id",
    "neighbourhood_group_c",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
    "price",
]

if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"{FILE_NAME} was not found. Upload it to the Google Colab workspace "
        "using the Files panel, then rerun this cell."
    )

df_raw = pd.read_csv(FILE_PATH)

missing_columns = [col for col in required_columns if col not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

# Keep only the fields required for validation, analysis, and modeling.
df = df_raw[required_columns].copy()

numeric_columns = [
    "id",
    "neighbourhood_group_c",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
    "price",
]

# Convert fields to numeric. Invalid values become missing and are removed below.
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_before = df.isna().sum()
duplicates_before = int(df.duplicated().sum())

# Remove records that cannot be used by the regression model.
df = df.dropna(subset=required_columns).drop_duplicates().copy()

# Validate the borough coding required by the case.
valid_borough_codes = {1, 2, 3, 4, 5}
invalid_codes = sorted(
    set(df["neighbourhood_group_c"].astype(int).unique()) - valid_borough_codes
)
if invalid_codes:
    raise ValueError(f"Invalid neighbourhood_group_c values found: {invalid_codes}")

borough_map = {
    1: "Manhattan",
    2: "Brooklyn",
    3: "Queens",
    4: "Staten Island",
    5: "Bronx",
}
borough_order = ["Manhattan", "Brooklyn", "Queens", "Staten Island", "Bronx"]
reference_borough = "Manhattan"

df["neighbourhood_group_c"] = df["neighbourhood_group_c"].astype(int)
df["borough"] = pd.Categorical(
    df["neighbourhood_group_c"].map(borough_map),
    categories=borough_order,
)
df["high_value_status"] = np.where(df["price"] > 120, "Above $120", "$120 or below")

print("Original shape:", df_raw.shape)
print("Modeling shape after validation:", df.shape)
print("\nMissing values before cleaning:")
display(missing_before.to_frame("missing_count"))
print("Duplicate rows before cleaning:", duplicates_before)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nPrepared data preview:")
display(df.head())

# id is deliberately excluded from the predictors because it is only a unique
# listing identifier and has no meaningful business interpretation.


# CELL 3 — Train Multiple Linear Regression and evaluate reliability

continuous_predictors = [
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
]
target_column = "price"

# Borough is categorical, so use one-hot/dummy encoding rather than the numeric
# 1-5 borough code. Manhattan is the reference category; each borough coefficient
# is interpreted relative to Manhattan.
borough_dummies = pd.get_dummies(
    df["borough"],
    prefix="borough",
    drop_first=True,
    dtype=float,
)

X = pd.concat(
    [
        df[continuous_predictors].astype(float),
        borough_dummies,
    ],
    axis=1,
)
predictor_columns = X.columns.tolist()
y = df[target_column].astype(float)

# 80% trains the model; 20% tests performance on unseen listings.
# Comparing training and test performance helps identify possible overfitting.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

model = LinearRegression()
model.fit(X_train, y_train)

train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

def adjusted_r2(r2_value, n_observations, n_predictors):
    """Calculate Adjusted R² for a specific dataset."""
    denominator = n_observations - n_predictors - 1
    if denominator <= 0:
        return np.nan
    return 1 - ((1 - r2_value) * (n_observations - 1) / denominator)

def calculate_metrics(actual, predicted, n_predictors):
    r2 = r2_score(actual, predicted)
    return {
        "R²": r2,
        "Adjusted R²": adjusted_r2(r2, len(actual), n_predictors),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAE": mean_absolute_error(actual, predicted),
    }

train_metrics = calculate_metrics(y_train, train_predictions, len(predictor_columns))
test_metrics = calculate_metrics(y_test, test_predictions, len(predictor_columns))

adjusted_r2_gap = abs(
    train_metrics["Adjusted R²"] - test_metrics["Adjusted R²"]
)

def reliability_assessment(train_adj, test_adj):
    gap = abs(train_adj - test_adj)

    if test_adj < 0:
        return (
            f"The test Adjusted R² is {test_adj:.3f}, which is below zero. "
            "The model does not generalize well enough to be treated as a reliable "
            "stand-alone pricing tool. It may still support exploration, but Joan "
            "should use substantial caution and additional market evidence."
        )

    if test_adj < 0.30:
        quality = (
            "The model explains only a limited portion of price variation in unseen listings. "
        )
    elif test_adj < 0.60:
        quality = (
            "The model provides moderate explanatory and predictive value for unseen listings. "
        )
    else:
        quality = (
            "The model provides relatively strong explanatory and predictive value for unseen listings. "
        )

    if gap <= 0.05:
        overfit = (
            f"The train-test Adjusted R² difference is {gap:.3f}, indicating little evidence "
            "of overfitting."
        )
    elif gap <= 0.10:
        overfit = (
            f"The train-test Adjusted R² difference is {gap:.3f}, indicating some performance "
            "decline and a need for caution."
        )
    else:
        overfit = (
            f"The train-test Adjusted R² difference is {gap:.3f}, indicating possible "
            "overfitting or weak generalization."
        )

    return (
        quality
        + overfit
        + " The prediction should support, not replace, Joan's business judgment."
    )

reliability_text = reliability_assessment(
    train_metrics["Adjusted R²"],
    test_metrics["Adjusted R²"],
)

coefficient_df = pd.DataFrame(
    {
        "Variable": predictor_columns,
        "Coefficient": model.coef_,
    }
)
coefficient_df["Direction"] = np.where(
    coefficient_df["Coefficient"] >= 0, "Positive", "Negative"
)

def change_word(value):
    return "increase" if value >= 0 else "decrease"

def business_interpretation(row):
    variable = row["Variable"]
    coefficient = row["Coefficient"]
    amount = abs(coefficient)

    continuous_text = {
        "minimum_nights": (
            f"One additional required minimum night is associated with an estimated "
            f"${amount:,.2f} {change_word(coefficient)} in nightly price, holding other variables constant."
        ),
        "number_of_reviews": (
            f"One additional total review is associated with an estimated "
            f"${amount:,.2f} {change_word(coefficient)} in nightly price, holding other variables constant."
        ),
        "reviews_per_month": (
            f"One additional review per month is associated with an estimated "
            f"${amount:,.2f} {change_word(coefficient)} in nightly price, holding other variables constant."
        ),
        "availability_365": (
            f"One additional available day per year is associated with an estimated "
            f"${amount:,.2f} {change_word(coefficient)} in nightly price, holding other variables constant."
        ),
    }

    if variable in continuous_text:
        return continuous_text[variable]

    if variable.startswith("borough_"):
        borough_name = variable.replace("borough_", "", 1)
        direction = "higher" if coefficient >= 0 else "lower"
        return (
            f"Compared with {reference_borough}, a listing in {borough_name} is associated with "
            f"an estimated ${amount:,.2f} {direction} nightly price, holding other variables constant."
        )

    return "Interpretation not available."

coefficient_df["Business interpretation"] = coefficient_df.apply(
    business_interpretation,
    axis=1,
)

largest_raw_row = coefficient_df.iloc[
    coefficient_df["Coefficient"].abs().argmax()
]
coef_map = dict(zip(predictor_columns, model.coef_))

borough_effect_lines = []
for borough_name in borough_order[1:]:
    feature_name = f"borough_{borough_name}"
    if feature_name in coef_map:
        coefficient = coef_map[feature_name]
        direction = "higher" if coefficient >= 0 else "lower"
        borough_effect_lines.append(
            f"- **{borough_name} vs {reference_borough}:** "
            f"${abs(coefficient):,.2f} {direction} expected nightly price, holding other variables constant."
        )

coefficient_summary = f"""
### Coefficient interpretation

- **Reference borough:** {reference_borough}. Borough coefficients below compare each borough with {reference_borough}.
- **Intercept:** ${model.intercept_:,.2f}. This is the model's baseline prediction for the reference borough when the numeric predictors equal zero; it may not describe a realistic listing.
- **Reviews per month:** One additional review per month is associated with a **${abs(coef_map["reviews_per_month"]):,.2f} {change_word(coef_map["reviews_per_month"])}** in expected nightly price, holding other variables constant.
- **Availability:** One additional available day is associated with a **${abs(coef_map["availability_365"]):,.2f} {change_word(coef_map["availability_365"])}** in expected nightly price, holding other variables constant.
- **Minimum nights:** One additional required night is associated with a **${abs(coef_map["minimum_nights"]):,.2f} {change_word(coef_map["minimum_nights"])}** in expected nightly price, holding other variables constant.

**Borough effects relative to {reference_borough}:**

{chr(10).join(borough_effect_lines)}

- **Largest absolute raw coefficient:** `{largest_raw_row["Variable"]}` at {largest_raw_row["Coefficient"]:.3f}.
- **Caution:** Raw coefficients use different measurement units, so coefficient size alone is not a perfect comparison of relative importance.
- **Interpretation note:** These coefficients describe associations in the historical data; they do not establish causation.
"""

metrics_table = pd.DataFrame(
    [
        ["Training", len(y_train), train_metrics["R²"], train_metrics["Adjusted R²"], train_metrics["RMSE"], train_metrics["MAE"]],
        ["Testing", len(y_test), test_metrics["R²"], test_metrics["Adjusted R²"], test_metrics["RMSE"], test_metrics["MAE"]],
    ],
    columns=["Dataset", "Records", "R²", "Adjusted R²", "RMSE", "MAE"],
)

print("Model performance:")
display(metrics_table.round(4))
print("\nReliability assessment:")
print(reliability_text)
print("\nCoefficients:")
display(coefficient_df)


# CELL 4 — Dashboard functions: filters, KPIs, charts, insights, and simulator

bounds = {
    col: (float(df[col].min()), float(df[col].max()))
    for col in [
        "minimum_nights",
        "number_of_reviews",
        "reviews_per_month",
        "availability_365",
        "price",
    ]
}

def empty_figure(title, message="No listings match the current filter selection."):
    fig = go.Figure()
    fig.add_annotation(
        text=message,
        x=0.5,
        y=0.5,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=16),
    )
    fig.update_layout(
        title=title,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        height=430,
    )
    return fig

def add_price_benchmark(fig):
    fig.add_hline(
        y=120,
        line_dash="dash",
        annotation_text="$120 benchmark",
        annotation_position="top left",
    )
    return fig

def add_linear_trend(fig, data, x_column, y_column="price"):
    """Add a simple overall least-squares trend line using NumPy only."""
    clean = data[[x_column, y_column]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean) >= 2 and clean[x_column].nunique() >= 2:
        slope, intercept = np.polyfit(clean[x_column], clean[y_column], 1)
        x_values = np.linspace(clean[x_column].min(), clean[x_column].max(), 100)
        y_values = slope * x_values + intercept
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=y_values,
                mode="lines",
                name="Overall linear trend",
                line=dict(width=3),
            )
        )
    return fig

def create_dashboard_figures(data):
    if data.empty:
        return tuple(
            empty_figure(title)
            for title in [
                "Nightly Price by Borough",
                "Reviews per Month vs Nightly Price",
                "Number of Reviews vs Nightly Price",
                "Minimum Nights vs Nightly Price",
                "Availability vs Nightly Price",
                "High-Value Listing Rate by Borough",
            ]
        )

    fig1 = px.box(
        data,
        x="borough",
        y="price",
        category_orders={"borough": borough_order},
        points="outliers",
        labels={"borough": "Borough", "price": "Nightly Price ($)"},
        title="Nightly Price by Borough",
    )
    add_price_benchmark(fig1)

    fig2 = px.scatter(
        data,
        x="reviews_per_month",
        y="price",
        color="borough",
        category_orders={"borough": borough_order},
        hover_data=["minimum_nights", "number_of_reviews", "availability_365"],
        labels={
            "reviews_per_month": "Reviews per Month",
            "price": "Nightly Price ($)",
            "borough": "Borough",
        },
        title="Reviews per Month vs Nightly Price",
        opacity=0.65
    )
    add_price_benchmark(fig2)
    add_linear_trend(fig2, data, "reviews_per_month")

    fig3 = px.scatter(
        data,
        x="number_of_reviews",
        y="price",
        color="borough",
        category_orders={"borough": borough_order},
        hover_data=["reviews_per_month", "minimum_nights", "availability_365"],
        labels={
            "number_of_reviews": "Number of Reviews",
            "price": "Nightly Price ($)",
            "borough": "Borough",
        },
        title="Number of Reviews vs Nightly Price",
        opacity=0.65
    )
    add_price_benchmark(fig3)
    add_linear_trend(fig3, data, "number_of_reviews")

    fig4 = px.scatter(
        data,
        x="minimum_nights",
        y="price",
        color="borough",
        category_orders={"borough": borough_order},
        hover_data=["number_of_reviews", "reviews_per_month", "availability_365"],
        labels={
            "minimum_nights": "Minimum Nights",
            "price": "Nightly Price ($)",
            "borough": "Borough",
        },
        title="Minimum Nights vs Nightly Price",
        opacity=0.65
    )
    add_price_benchmark(fig4)
    add_linear_trend(fig4, data, "minimum_nights")

    fig5 = px.scatter(
        data,
        x="availability_365",
        y="price",
        color="borough",
        category_orders={"borough": borough_order},
        hover_data=["minimum_nights", "number_of_reviews", "reviews_per_month"],
        labels={
            "availability_365": "Availability (Days per Year)",
            "price": "Nightly Price ($)",
            "borough": "Borough",
        },
        title="Availability vs Nightly Price",
        opacity=0.65
    )
    add_price_benchmark(fig5)
    add_linear_trend(fig5, data, "availability_365")

    borough_rate = (
        data.assign(high_value=(data["price"] > 120).astype(int))
        .groupby("borough", observed=False)["high_value"]
        .mean()
        .mul(100)
        .reindex(borough_order)
        .dropna()
        .reset_index(name="percent_above_120")
    )
    fig6 = px.bar(
        borough_rate,
        x="borough",
        y="percent_above_120",
        category_orders={"borough": borough_order},
        text_auto=".1f",
        labels={
            "borough": "Borough",
            "percent_above_120": "Listings Above $120 (%)",
        },
        title="High-Value Listing Rate by Borough",
    )
    fig6.update_yaxes(range=[0, max(100, float(borough_rate["percent_above_120"].max()) * 1.15)])

    for fig in [fig1, fig2, fig3, fig4, fig5, fig6]:
        fig.update_layout(height=430, margin=dict(l=30, r=30, t=60, b=40))

    return fig1, fig2, fig3, fig4, fig5, fig6

def answer_business_questions(data):
    if data.empty:
        return "### Answers to Joan's Business Questions\nNo listings match the current filters."

    borough_summary = (
        data.groupby("borough", observed=False)["price"]
        .median()
        .sort_values(ascending=False)
    )
    top_borough = borough_summary.index[0]
    top_borough_price = borough_summary.iloc[0]

    correlations = data[
        ["price", "number_of_reviews", "reviews_per_month", "minimum_nights", "availability_365"]
    ].corr(numeric_only=True)["price"]

    def direction(value):
        if abs(value) < 0.10:
            return "very weak"
        if abs(value) < 0.30:
            return "weak positive" if value > 0 else "weak negative"
        if abs(value) < 0.50:
            return "moderate positive" if value > 0 else "moderate negative"
        return "strong positive" if value > 0 else "strong negative"

    return f"""
### Answers to Joan's Business Questions

1. **Which boroughs are associated with higher prices?**
   In the current selection, **{top_borough}** has the highest median nightly price at **${top_borough_price:,.2f}**.

2. **Are reviews associated with higher prices?**
   The correlation between total reviews and price is **{correlations["number_of_reviews"]:.3f}** ({direction(correlations["number_of_reviews"])}), while reviews per month and price have a correlation of **{correlations["reviews_per_month"]:.3f}** ({direction(correlations["reviews_per_month"])}).

3. **Do minimum-night requirements influence price?**
   Minimum nights and price have a correlation of **{correlations["minimum_nights"]:.3f}** ({direction(correlations["minimum_nights"])}). The regression coefficient gives the adjusted association while holding the other predictors constant.

4. **Does availability affect price?**
   Availability and price have a correlation of **{correlations["availability_365"]:.3f}** ({direction(correlations["availability_365"])}). The model estimates a coefficient of **{model.coef_[4]:.3f}** dollars per additional available day, holding the other predictors constant.

These results describe associations in the available data and do not prove causation.
"""

def filter_data(
    boroughs,
    minimum_nights_range,
    reviews_range,
    reviews_per_month_range,
    availability_range,
    price_range,
    high_value_status,
):
    filtered = df.copy()

    if boroughs:
        filtered = filtered[filtered["borough"].isin(boroughs)]

    filtered = filtered[
        filtered["minimum_nights"].between(*minimum_nights_range)
        & filtered["number_of_reviews"].between(*reviews_range)
        & filtered["reviews_per_month"].between(*reviews_per_month_range)
        & filtered["availability_365"].between(*availability_range)
        & filtered["price"].between(*price_range)
    ]

    if high_value_status == "Above $120":
        filtered = filtered[filtered["price"] > 120]
    elif high_value_status == "$120 or below":
        filtered = filtered[filtered["price"] <= 120]

    return filtered

def dashboard_update(
    boroughs,
    minimum_nights_range,
    reviews_range,
    reviews_per_month_range,
    availability_range,
    price_range,
    high_value_status,
):
    filtered = filter_data(
        boroughs,
        minimum_nights_range,
        reviews_range,
        reviews_per_month_range,
        availability_range,
        price_range,
        high_value_status,
    )

    if filtered.empty:
        kpis = ("0", "N/A", "N/A", "N/A", "N/A")
        status = "⚠️ No listings match the current filter selection. Please adjust or reset the filters."
    else:
        kpis = (
            f"{len(filtered):,}",
            f"${filtered['price'].mean():,.2f}",
            f"${filtered['price'].median():,.2f}",
            f"{(filtered['price'].gt(120).mean() * 100):.1f}%",
            f"${filtered['price'].max():,.2f}",
        )
        status = f"Showing {len(filtered):,} of {len(df):,} listings."

    figures = create_dashboard_figures(filtered)
    answers = answer_business_questions(filtered)

    return (*kpis, status, *figures, answers)

def dashboard_update_from_numbers(
    boroughs,
    minimum_nights_min, minimum_nights_max,
    reviews_min, reviews_max,
    reviews_month_min, reviews_month_max,
    availability_min, availability_max,
    price_min, price_max,
    high_value_status,
):
    return dashboard_update(
        boroughs,
        [minimum_nights_min, minimum_nights_max],
        [reviews_min, reviews_max],
        [reviews_month_min, reviews_month_max],
        [availability_min, availability_max],
        [price_min, price_max],
        high_value_status,
    )

def reset_dashboard():
    defaults = (
        borough_order,
        bounds["minimum_nights"][0], bounds["minimum_nights"][1],
        bounds["number_of_reviews"][0], bounds["number_of_reviews"][1],
        bounds["reviews_per_month"][0], bounds["reviews_per_month"][1],
        bounds["availability_365"][0], bounds["availability_365"][1],
        bounds["price"][0], bounds["price"][1],
        "All listings",
    )
    outputs = dashboard_update_from_numbers(*defaults)
    return (*defaults, *outputs)

def estimate_price(borough, minimum_nights, number_of_reviews, reviews_per_month, availability_365):
    if borough not in borough_order:
        return "Please select a valid borough.", go.Figure()

    numeric_values = {
        "minimum_nights": minimum_nights,
        "number_of_reviews": number_of_reviews,
        "reviews_per_month": reviews_per_month,
        "availability_365": availability_365,
    }

    for variable, value in numeric_values.items():
        if value is None or not np.isfinite(float(value)):
            return f"Please enter a valid value for {variable}.", go.Figure()
        low, high = bounds[variable]
        if not (low <= float(value) <= high):
            return (
                f"{variable} must be between {low:,.2f} and {high:,.2f}.",
                go.Figure(),
            )

    # Build the simulator row with the same one-hot encoding used to train the model.
    input_base = pd.DataFrame(
        [{
            "borough": borough,
            "minimum_nights": float(minimum_nights),
            "number_of_reviews": float(number_of_reviews),
            "reviews_per_month": float(reviews_per_month),
            "availability_365": float(availability_365),
        }]
    )
    input_base["borough"] = pd.Categorical(
        input_base["borough"],
        categories=borough_order,
    )
    input_borough_dummies = pd.get_dummies(
        input_base["borough"],
        prefix="borough",
        drop_first=True,
        dtype=float,
    )
    input_row = pd.concat(
        [
            input_base[continuous_predictors].astype(float),
            input_borough_dummies,
        ],
        axis=1,
    ).reindex(columns=predictor_columns, fill_value=0.0)

    predicted_price = float(model.predict(input_row)[0])
    difference = predicted_price - 120

    if predicted_price > 120:
        classification = "Potentially High-Value Opportunity"
        comparison = f"${difference:,.2f} above"
    else:
        classification = "Predicted Price Does Not Exceed $120"
        comparison = f"${abs(difference):,.2f} below"

    result = f"""
### Estimated Result

**Predicted nightly price:** ${predicted_price:,.2f}
**Classification:** {classification}
**Benchmark comparison:** {comparison} the $120 benchmark.

The model estimate is based on listings with similar measured characteristics. Borough is treated as a categorical variable, with {reference_borough} as the reference category.
*This estimate is based on historical listing patterns and should support, not replace, business judgment.*
"""

    upper_limit = max(150, predicted_price * 1.25, float(df["price"].quantile(0.95)))
    gauge = go.Figure(
        go.Indicator(
            mode="gauge+number+delta",
            value=predicted_price,
            number={"prefix": "$", "valueformat": ",.2f"},
            delta={"reference": 120, "prefix": "$", "valueformat": ",.2f"},
            title={"text": "Predicted Nightly Price vs $120 Benchmark"},
            gauge={
                "axis": {"range": [0, upper_limit]},
                "threshold": {
                    "line": {"width": 4},
                    "thickness": 0.8,
                    "value": 120,
                },
            },
        )
    )
    gauge.update_layout(height=350, margin=dict(l=35, r=35, t=70, b=30))

    return result, gauge

def reset_simulator():
    return (
        "Manhattan",
        int(df["minimum_nights"].median()),
        int(df["number_of_reviews"].median()),
        float(df["reviews_per_month"].median()),
        int(df["availability_365"].median()),
        "Enter listing characteristics and select **Estimate Nightly Price**.",
        go.Figure(),
    )

initial_figures = create_dashboard_figures(df)
initial_answers = answer_business_questions(df)


# CELL 5 — Build and launch the Gradio Blocks application

css = """
.gradio-container {max-width: 1500px !important;}
.kpi-card textarea {font-size: 20px !important; font-weight: 700 !important; text-align: center;}
"""

with gr.Blocks(title="Airbnb Rental Arbitrage Analytics", css=css) as demo:
    gr.Markdown(
        """
        # Airbnb Rental Arbitrage: Nightly Price Analytics
        **Decision-support dashboard for identifying higher-value New York City Airbnb opportunities**

        Listings priced **above $120 per night** are treated as potentially higher-value opportunities in this case.
        """
    )

    with gr.Tabs():
        with gr.Tab("Exploration Dashboard"):
            gr.Markdown("## Filters")
            with gr.Row():
                borough_filter = gr.Dropdown(
                    choices=borough_order,
                    value=borough_order,
                    multiselect=True,
                    label="Borough",
                )
                high_value_filter = gr.Radio(
                    choices=["All listings", "Above $120", "$120 or below"],
                    value="All listings",
                    label="High-Value Status",
                )

            with gr.Row():
                minimum_nights_min = gr.Number(value=bounds["minimum_nights"][0], label="Minimum Nights — Min")
                minimum_nights_max = gr.Number(value=bounds["minimum_nights"][1], label="Minimum Nights — Max")
                reviews_min = gr.Number(value=bounds["number_of_reviews"][0], label="Number of Reviews — Min")
                reviews_max = gr.Number(value=bounds["number_of_reviews"][1], label="Number of Reviews — Max")

            with gr.Row():
                reviews_month_min = gr.Number(value=bounds["reviews_per_month"][0], label="Reviews per Month — Min")
                reviews_month_max = gr.Number(value=bounds["reviews_per_month"][1], label="Reviews per Month — Max")
                availability_min = gr.Number(value=bounds["availability_365"][0], label="Availability — Min")
                availability_max = gr.Number(value=bounds["availability_365"][1], label="Availability — Max")

            with gr.Row():
                price_min = gr.Number(value=bounds["price"][0], label="Nightly Price — Min ($)")
                price_max = gr.Number(value=bounds["price"][1], label="Nightly Price — Max ($)")

            with gr.Row():
                apply_filters_button = gr.Button("Apply Filters", variant="primary")
                reset_filters_button = gr.Button("Reset Filters")

            status_message = gr.Markdown(f"Showing {len(df):,} of {len(df):,} listings.")

            gr.Markdown("## KPI Summary")
            with gr.Row():
                listings_kpi = gr.Textbox(value=f"{len(df):,}", label="Listings", interactive=False, elem_classes="kpi-card")
                average_price_kpi = gr.Textbox(value=f"${df['price'].mean():,.2f}", label="Average Price", interactive=False, elem_classes="kpi-card")
                median_price_kpi = gr.Textbox(value=f"${df['price'].median():,.2f}", label="Median Price", interactive=False, elem_classes="kpi-card")
                above_120_kpi = gr.Textbox(value=f"{df['price'].gt(120).mean() * 100:.1f}%", label="Above $120", interactive=False, elem_classes="kpi-card")
                highest_price_kpi = gr.Textbox(value=f"${df['price'].max():,.2f}", label="Highest Price", interactive=False, elem_classes="kpi-card")

            gr.Markdown("## Interactive Visualizations")
            with gr.Row():
                borough_plot = gr.Plot(value=initial_figures[0])
                reviews_month_plot = gr.Plot(value=initial_figures[1])
            with gr.Row():
                reviews_plot = gr.Plot(value=initial_figures[2])
                minimum_nights_plot = gr.Plot(value=initial_figures[3])
            with gr.Row():
                availability_plot = gr.Plot(value=initial_figures[4])
                high_value_plot = gr.Plot(value=initial_figures[5])

            business_answers = gr.Markdown(value=initial_answers)

        with gr.Tab("Model Performance"):
            gr.Markdown("## Model Performance and Reliability")
            gr.Dataframe(value=metrics_table.round(4), interactive=False, label="Training and Test Metrics")
            gr.Markdown(
                f"""
                **Adjusted R² train-test difference:** {adjusted_r2_gap:.4f}

                **Reliability assessment:** {reliability_text}
                """
            )

            gr.Markdown("## Model Coefficients and Business Interpretation")
            gr.Dataframe(
                value=coefficient_df.round({"Coefficient": 4}),
                interactive=False,
                wrap=True,
                label="Regression Coefficients",
            )
            gr.Markdown(coefficient_summary)

        with gr.Tab("Price Simulator"):
            gr.Markdown(
                """
                ## Potential Listing Price Simulator
                Enter the characteristics of a potential listing and compare its predicted price with the **$120 benchmark**.
                """
            )
            with gr.Row():
                simulator_borough = gr.Dropdown(choices=borough_order, value="Manhattan", label="Borough")
                simulator_minimum_nights = gr.Number(value=int(df["minimum_nights"].median()), label="Minimum Nights")
                simulator_reviews = gr.Number(value=int(df["number_of_reviews"].median()), label="Number of Reviews")

            with gr.Row():
                simulator_reviews_month = gr.Number(value=float(df["reviews_per_month"].median()), label="Reviews per Month")
                simulator_availability = gr.Number(value=int(df["availability_365"].median()), label="Availability (Days per Year)")

            with gr.Row():
                estimate_button = gr.Button("Estimate Nightly Price", variant="primary")
                reset_simulator_button = gr.Button("Reset Simulator")

            simulator_result = gr.Markdown("Enter listing characteristics and select **Estimate Nightly Price**.")
            simulator_gauge = gr.Plot(value=go.Figure())

    filter_inputs = [
        borough_filter,
        minimum_nights_min, minimum_nights_max,
        reviews_min, reviews_max,
        reviews_month_min, reviews_month_max,
        availability_min, availability_max,
        price_min, price_max,
        high_value_filter,
    ]

    dashboard_outputs = [
        listings_kpi,
        average_price_kpi,
        median_price_kpi,
        above_120_kpi,
        highest_price_kpi,
        status_message,
        borough_plot,
        reviews_month_plot,
        reviews_plot,
        minimum_nights_plot,
        availability_plot,
        high_value_plot,
        business_answers,
    ]

    apply_filters_button.click(
        fn=dashboard_update_from_numbers,
        inputs=filter_inputs,
        outputs=dashboard_outputs,
    )

    reset_filters_button.click(
        fn=reset_dashboard,
        inputs=None,
        outputs=filter_inputs + dashboard_outputs,
    )

    estimate_button.click(
        fn=estimate_price,
        inputs=[
            simulator_borough,
            simulator_minimum_nights,
            simulator_reviews,
            simulator_reviews_month,
            simulator_availability,
        ],
        outputs=[simulator_result, simulator_gauge],
    )

    reset_simulator_button.click(
        fn=reset_simulator,
        inputs=None,
        outputs=[
            simulator_borough,
            simulator_minimum_nights,
            simulator_reviews,
            simulator_reviews_month,
            simulator_availability,
            simulator_result,
            simulator_gauge,
        ],
    )

demo.launch(share=True, debug=False)
